In [ ]:
from pathlib import Path
import os, sys, sqlite3, pandas as pd

PROSJEKT = Path.cwd()
if PROSJEKT.name == "notebooks":
    PROSJEKT = PROSJEKT.parent
os.chdir(PROSJEKT)
if str(PROSJEKT) not in sys.path:
    sys.path.insert(0, str(PROSJEKT))

from src.kunstpixel.core import db  # Ferdige fasit-kalkulatorer (lønn, overtid, sykefravær)

con = sqlite3.connect("file:05_data/museum.db?mode=ro", uri=True)   # mode=ro: kan ikke endre databasen
sql = lambda q: pd.read_sql(q, con)
sql("select name from sqlite_master where type='table' order by name")

In [ ]:
df = sql("""
select d.maaned, sum(d.besok_totalt) as besok, v.vakter, v.timer
from dagsbesok d
join (select maaned, count(*) as vakter, sum(timer) as timer from hr_vakter where aar = 2026 group by maaned) v using (maaned)
where d.aar = 2026 and d.er_aapent = 1
group by d.maaned order by d.maaned
""")
df["besok_per_vakttime"] = (df.besok / df.timer).round(1)
df.plot(x="maaned", y="besok_per_vakttime", kind="bar", title="Besøk per vakttime i 2026")
df

In [ ]:
# Bruker samme fasit-kalkulator som nettsiden (Datafinner), slik at kronebeløpene alltid er 100 % like
kost = db.query_shift_cost("2026-05")
df_kost = pd.DataFrame(kost["avdelinger"])
df_kost